In [1]:
from langchain.document_loaders import PyPDFLoader 
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from langchain_ollama.embeddings import OllamaEmbeddings 
from langchain_chroma import Chroma
from langchain_groq import ChatGroq 

In [2]:
reader = PyPDFLoader("./content/Bpifrance Creation_GUIDE PRATIQUE DU CREATEUR_2019.pdf") 
doc = reader.load()

In [3]:
text_splitter = RecursiveCharacterTextSplitter( chunk_size=1000,chunk_overlap=200, ) 
chunks=text_splitter.split_documents(doc)

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

'sh' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [7]:
!ollama pull nomic-embed-text

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 970aa74c0a90:   0% ▕                  ▏ 361 KB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   0% ▕                  ▏ 920 KB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 1.4 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 2.2 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 2.6 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 2.6 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 2.6 MB/274

In [8]:
embeddingModel = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")

In [9]:
dbVector = Chroma( persist_directory="./dbVector", embedding_function=embeddingModel,)

In [10]:
dbVector.add_documents(chunks)

['24eca8e2-d8af-435f-90f3-f566b8731825',
 'a382f5fa-f8d4-4cb3-b1f9-3c547b918959',
 'f01f2b6b-e2f9-48ef-8e42-3143a1249ca4',
 '1cce314b-1498-4765-a2b7-b9edbeaa1456',
 'f7355186-18e2-41f5-867d-4c42d05182c6',
 'cda924b9-7a3d-4ec2-919c-d055d6b8eae3',
 '03203008-6622-4f4f-b22b-f99c981cce68',
 'b61a919d-a71f-432b-9cbc-dd829f58328b',
 '34e528b9-6a25-48e9-86d4-51812b1c5ed4',
 '8647f5ef-5e71-454d-9e6a-d9472f4a3930',
 'aa4e272a-ebc3-414b-939e-ef0eb61483ba',
 '90148bbb-4d24-4eaf-a305-4502e3176c34',
 'a7b267d0-ccb9-4037-92ad-05e7de48f713',
 '8da21b5b-8912-4b11-9e67-7aacd5f475d1',
 '5e8ae6a9-d2b7-47bf-ab10-5e49ea6a7a78',
 'c78ce328-6b74-4602-927d-1c73d68b0d22',
 'ddc1a49a-6b17-44a0-9a87-7cc98e18b473',
 'a8de606d-f4dc-42d0-a3ee-58f135d469c7',
 'bd1f2348-67ef-4529-bed8-1a668d49ec92',
 'd4acf587-4c23-45ec-9f43-eab87d83761c',
 '7692da5a-a2d8-403f-86ff-ef798cc8ef3c',
 'a6344231-0664-4fb0-b1e8-f87aba41f707',
 '2a31e73c-d7e2-403b-8fc1-9e11bfbe7b1c',
 '97f95105-d3a3-4db5-b495-4eaa0713f2e2',
 '5aae18a9-12c2-

In [11]:
llm = ChatGroq( base_url="https://api.groq.com", model="meta-llama/llama-4-maverick-17b-128e-instruct", api_key="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", )

In [12]:
llm.invoke("Bonjour, je suis Youssef et vous ??").content

"Bonjour Youssef ! Je suis un modèle de langage basé sur l'intelligence artificielle, conçu pour comprendre et répondre à vos questions et commentaires. Je n'ai pas de nom personnel, mais vous pouvez me considérer comme votre assistant virtuel. Comment puis-je vous aider aujourd'hui ?"

In [13]:
def serachSimilarty(question):
  contexteSimilaire=dbVector.similarity_search(question)
  texts= ""
  for mor in contexteSimilaire:
    texts += texts + mor.page_content
  return texts

In [14]:
def prompt(question,llm):
  contexteSimilaire=serachSimilarty(question)
  response=llm.invoke(f"Repondez à cette question {question} en se basant sur le contexte suivant {contexteSimilaire}")
  return response.content

In [15]:
prompt("Quelles sont les étapes pour créer une entreprise ?",llm)

"## Étapes pour créer une entreprise\n\n1. **Trouver une idée de création d'entreprise** : \n   - Exploitez votre propre idée ou valorisez celle des autres.\n   - Vérifiez si cette idée présente de réels débouchés économiques.\n   - Transformez cette idée en projet.\n\n2. **Développer votre projet** :\n   - Reprenez ou créez une entreprise.\n   - Soyez prudent avec les idées qui pourraient ne rapporter qu'à ceux qui les ont conçues pour les vendre.\n\n3. **Financer votre entreprise** :\n   - Utilisez l’autofinancement.\n   - Recherchez des subventions (consultez la Base nationale des aides aux entreprises).\n   - Ouvrez le capital de votre entreprise à des investisseurs (si vous créez une société).\n   - Sollicitez un ou plusieurs emprunts.\n\n4. **Mettre en œuvre votre projet** :\n   - Préparez votre plan de financement.\n   - Présentez votre entreprise et rencontrez des partenaires commerciaux.\n\n## Facteurs clés de succès\nLes facteurs clés de succès d’un projet de création d’entre